In [1]:
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

C:\Users\LENOVO\Desktop\Mini_Foresight\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\LENOVO\.cache\kagglehub\datasets\ealaxi\paysim1\versions\2


In [2]:
import pandas as pd
file_path = os.path.join(path, "PS_20174392719_1491204439457_log.csv")

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


| Column | Description |
|---|---|
| step | Transaction time (1 step = 1 hour) |
| type | Type of transaction |
| amount | Transaction amount |
| nameOrig | Sender's account ID |
| oldbalanceOrg | Sender's balance before transaction |
| newbalanceOrig | Sender's balance after transaction |
| nameDest | Recipient's account ID |
| oldbalanceDest | Recipient's balance before transaction |
| newbalanceDest | Recipient's balance after transaction |
| isFraud | Actual fraud indicator (0 or 1) |
| isFlaggedFraud | System-flagged fraud indicator (0 or 1) |

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [4]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [5]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [6]:
print("Transaction Types:")
print(df['type'].value_counts())

print("\nFraud Distribution:")
print(df['isFraud'].value_counts())

Transaction Types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

Fraud Distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [7]:
fraud_by_type = df.groupby('type')['isFraud'].agg(
    ['count', 'sum', 'mean']
)

fraud_by_type.columns = [
    'total_transactions',
    'fraud_count',
    'fraud_rate'
]

fraud_by_type.sort_values(
    by='fraud_count',
    ascending=False
)

,total_transactions,fraud_count,fraud_rate
type,,,
CASH_OUT,2237500,4116,0.001840
TRANSFER,532909,4097,0.007688
CASH_IN,1399284,0,0.000000
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000


## Əməliyyat növləri üzrə Fraud analizi

Bu mərhələdə fraud əməliyyatlarının müxtəlif
transaction type-lar üzrə paylanması araşdırılır.

Hər əməliyyat növü üçün aşağıdakı göstəricilər
hesablanır:

- Ümumi əməliyyat sayı.
- Fraud əməliyyatlarının sayı.
- Fraud əməliyyatlarının faiz göstəricisi.

Məqsəd fraud hallarının hansı əməliyyat növlərində
daha çox müşahidə olunduğunu müəyyən etməkdir.

## Əməliyyat növləri üzrə Fraud nəticələri

Analiz nəticəsində fraud əməliyyatlarının yalnız
CASH_OUT və TRANSFER növlərində müşahidə edildiyi
müəyyən edildi.

- CASH_OUT: 4,116 fraud əməliyyatı (0.184%).
- TRANSFER: 4,097 fraud əməliyyatı (0.769%).
- CASH_IN, DEBIT və PAYMENT: 0 fraud əməliyyatı.

TRANSFER əməliyyatlarında fraud faizi CASH_OUT
əməliyyatlarından təxminən 4.2 dəfə yüksəkdir.

Bu müşahidələr gələcək Feature Engineering
mərhələsində əməliyyat növlərinin nəzərə alınması
üçün faydalıdır.

In [8]:
df['event_timestamp'] = (
    pd.Timestamp('2023-01-01') +
    pd.to_timedelta(df['step'] - 1, unit='h')
)

df[['step', 'event_timestamp']].head()

,step,event_timestamp
0,1,2023-01-01
1,1,2023-01-01
2,1,2023-01-01
3,1,2023-01-01
4,1,2023-01-01


## Event Timestamp yaradılması

Əməliyyatların zaman ardıcıllığını qorumaq və
Feature Store-da zaman əsaslı sorğuları dəstəkləmək
üçün `step` sütunundan timestamp yaradıldı.

Texniki başlanğıc tarixindən istifadə edilərək
əməliyyatlara tarix və saat məlumatı əlavə edildi.

In [9]:
df = df.sort_values(['nameOrig', 'event_timestamp']).copy()

## Feature Engineering

Bu mərhələdə müştərilərin əməliyyat tarixçəsinə əsaslanan feature-lər yaradılır. Məqsəd hər müştəri üçün əvvəlki əməliyyatların sayını və məbləğ göstəricilərini hesablamaqdır.

In [10]:
df['transaction_count'] = df.groupby('nameOrig').cumcount()
df[['nameOrig', 'step', 'transaction_count']].head(10)

,nameOrig,step,transaction_count
3196942,C1000000639,249,0
2798983,C1000001337,217,0
1013585,C1000001725,46,0
2980283,C1000002591,231,0
1920204,C1000003372,167,0
3374567,C1000003615,254,0
4548211,C1000004053,327,0
847860,C1000004530,41,0
2825292,C1000005353,226,0
2202461,C1000005555,185,0


In [11]:
df['total_amount'] = df.groupby("nameOrig")['amount'].cumsum() - df['amount']

In [12]:
df['avg_amount'] = (
    df['total_amount'] /
    df['transaction_count'].replace(0, float('nan'))
)

In [13]:
df[['transaction_count', 'total_amount', 'avg_amount']].isnull().sum()

transaction_count          0
total_amount               0
avg_amount           6353307
dtype: int64

## Feast Feature Store Setup

Feast kitabxanası vasitəsilə Feature Store qurulur. Müştəri Entity, məlumat mənbəyi (FileSource) və FeatureView təyin edilir.

In [14]:
import sys
import shutil

repo_path = r"C:\Users\LENOVO\Downloads\Devlab\paysim_feature_store"
repo_inner = os.path.join(repo_path, "feature_repo")

definitions_path = os.path.join(
    repo_inner, "feature_definitions.py"
)

In [15]:
feature_df = df[['nameOrig', 'event_timestamp', 'transaction_count', 'total_amount', 'avg_amount']]

In [16]:
%pip install feast

Note: you may need to restart the kernel to use updated packages.


In [17]:
import feast

print("Feast version:", feast.__version__)

Feast version: 0.67.0


In [18]:
data_path = os.path.join(repo_inner, 'data')

os.makedirs(data_path, exist_ok=True)

parquet_path = os.path.join(data_path, 'paysim_features.parquet')

feature_df.to_parquet(parquet_path, index=False)

print("Saved:", parquet_path)

Saved: C:\Users\LENOVO\Downloads\Devlab\paysim_feature_store\feature_repo\data\paysim_features.parquet


In [19]:
from pathlib import Path

feast_cli = Path(sys.executable).parent / "feast.exe"

if not feast_cli.exists():
    raise FileNotFoundError(f"Feast CLI not found: {feast_cli}")

features_path = parquet_path

print("Feast CLI:", feast_cli)
print("Feature file:", features_path)

Feast CLI: C:\Users\LENOVO\Desktop\Mini_Foresight\venv\Scripts\feast.exe
Feature file: C:\Users\LENOVO\Downloads\Devlab\paysim_feature_store\feature_repo\data\paysim_features.parquet


In [20]:
definitions_code = '''
from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource, ValueType
from feast.types import Int64, Float64


# 1. Entity
customer = Entity(
    name="customer",
    join_keys=["nameOrig"],
    value_type=ValueType.STRING,
    description="Customer identifier"
)


# 2. Data Source
customer_source = FileSource(
    name="paysim_features_source",
    path="data/paysim_features.parquet",
    timestamp_field="event_timestamp"
)


# 3. Feature View
customer_features = FeatureView(
    name="customer_transaction_features",
    entities=[customer],
    ttl=timedelta(days=365),
    schema=[
        Field(name="transaction_count", dtype=Int64),
        Field(name="total_amount", dtype=Float64),
        Field(name="avg_amount", dtype=Float64)
    ],
    source=customer_source,
    online=True,
    description="Historical transaction features for each customer"
)
'''

with open(definitions_path, 'w') as file:
    file.write(definitions_code)

print("Feature definitions saved successfully!")

Feature definitions saved successfully!


## Feature Registration

`feast apply` əmri ilə Entity və FeatureView Feast Registry-də qeydiyyata alınır.

In [21]:
import subprocess

result = subprocess.run(
    [str(feast_cli), "apply"],
    cwd=repo_inner,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

No project found in the repository. Using project name paysim_feature_store defined in feature_store.yaml
Applying changes for project paysim_feature_store
Updated feature view customer_transaction_features
	batch_source: type: BATCH_FILE
timestamp_field: "event_timestamp"
file_options {
  uri: "data/paysim_features.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "paysim_features_source"
meta {
  created_timestamp {
    seconds: 1793996676
    nanos: 308837000
  }
  last_updated_timestamp {
    seconds: 1793996676
    nanos: 308837000
  }
}
 -> type: BATCH_FILE
timestamp_field: "event_timestamp"
file_options {
  uri: "data/paysim_features.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "paysim_features_source"
meta {
  created_timestamp {
    seconds: 1794033880
    nanos: 649717000
  }
  last_updated_timestamp {
    seconds: 1794033880
    nanos: 649717000
  }
}


No changes to infrastructure




## Online Feature Retrieval

Online Store-dan seçilmiş müştərinin son feature-ləri əldə edilir. Bu, real-time serving prosesini nümayiş etdirir.

In [22]:
from feast import FeatureStore

store = FeatureStore(repo_path=repo_inner)

print("Feature Store initialized successfully!")

Feature Store initialized successfully!


In [23]:
store.registry.serve_features_while_materializing = True
store.registry.commit()

print(
    "Serving while materializing:",
    store.registry.serve_features_while_materializing
)

Serving while materializing: True


In [24]:
# Point-in-Time Retrieval

entity_df = feature_df[
    ['nameOrig', 'event_timestamp']
].sample(n=1000, random_state=42).copy()

historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "customer_transaction_features:transaction_count",
        "customer_transaction_features:total_amount",
        "customer_transaction_features:avg_amount"
    ]
).to_df()

print("Historical Features Shape:", historical_features.shape)

historical_features.head()

Historical Features Shape: (1000, 5)


,nameOrig,event_timestamp,transaction_count,total_amount,avg_amount
0,C2053491093,2023-01-01 02:00:00+00:00,0,0.0,NaN
1,C1933032750,2023-01-01 05:00:00+00:00,0,0.0,NaN
2,C1682454927,2023-01-01 06:00:00+00:00,0,0.0,NaN
3,C1360071225,2023-01-01 06:00:00+00:00,0,0.0,NaN
4,C1442039144,2023-01-01 07:00:00+00:00,0,0.0,NaN


##  Point-in-Time Retrieval (Zamana Əsaslanan Tarixi Məlumatların Əldə Edilməsi)

Bu mərhələdə 1000 əməliyyat nümunəsi seçilir və hər əməliyyat üçün həmin vaxta qədər mövcud olan müştəri göstəriciləri Feast vasitəsilə əldə edilir. Bu yanaşma gələcək məlumatların model təliminə sızmasının qarşısını alır.

In [27]:
from datetime import timedelta

materialization_start = pd.Timestamp("2023-01-01 02:00:00")
materialization_end = materialization_start + timedelta(hours=1)

hour_data = feature_df[
    (feature_df["event_timestamp"] >= materialization_start) &
    (feature_df["event_timestamp"] < materialization_end)
]

print("Transactions:", len(hour_data))
print("Unique customers:", hour_data["nameOrig"].nunique())
print("Start:", materialization_start)
print("End:", materialization_end)

Transactions: 552
Unique customers: 552
Start: 2023-01-01 02:00:00
End: 2023-01-01 03:00:00


## Online Store üçün Məlumatların Hazırlanması

Bütün dataset əvəzinə materialization prosesini nümayiş etdirmək üçün bir saatlıq zaman intervalı seçilir. Bu intervalda 552 əməliyyat və 552 unikal müştəri mövcuddur.

In [28]:
store.materialize(
    start_date=materialization_start,
    end_date=materialization_end
)

print("Materialization completed!")

Materializing 1 feature views from 2023-01-01 02:00:00+00:00 to 2023-01-01 03:00:00+00:00 into the sqlite online store.

customer_transaction_features:
Materialization completed!


###  Feature Materialization

Seçilmiş zaman intervalına aid feature-lər Offline Store-dan SQLite əsaslı Online Store-a köçürülür. Bu proses feature-lərin sürətli əldə edilməsi üçün onları Online Store-da əlçatan edir.

In [29]:
fv = store.get_feature_view("customer_transaction_features")

print("FeatureView state:", fv.state)
print("Last materialization:", fv.most_recent_end_time)

FeatureView state: 4
Last materialization: 2023-01-02 15:00:00+00:00


###  Materialization Statusunun Yoxlanılması

Materialization prosesindən sonra FeatureView-in vəziyyəti yoxlanılır. `AVAILABLE_ONLINE` statusu feature-lərin Online Store vasitəsilə əldə edilə bildiyini göstərir.

In [30]:
customer_id = hour_data.iloc[0]["nameOrig"]

print("Test customer:", customer_id)

Test customer: C1002872041


##  Online Feature Retrieval

Materialize edilmiş məlumatlar arasından test müştərisi seçilir. Onun müştəri identifikatoru vasitəsilə Online Store-dan feature-lər əldə edilir.

In [31]:
online_result = store.get_online_features(
    features=[
        "customer_transaction_features:transaction_count",
        "customer_transaction_features:total_amount",
        "customer_transaction_features:avg_amount"
    ],
    entity_rows=[
        {"nameOrig": customer_id}
    ]
).to_df()

online_result

,nameOrig,total_amount,avg_amount,transaction_count
0,C1002872041,0.0,None,0


In [32]:
offline_result = feature_df[
    (feature_df["nameOrig"] == customer_id) &
    (feature_df["event_timestamp"] == materialization_start)
]

offline_result

,nameOrig,event_timestamp,transaction_count,total_amount,avg_amount
3961,C1002872041,2023-01-01 02:00:00,0,0.0,NaN


##  Offline Feature Retrieval

Eyni müştərinin seçilmiş əməliyyat vaxtına uyğun feature-ləri Offline Store-dan əldə edilir. Bu nəticələr Online Store-dan alınmış göstəricilərlə müqayisə üçün istifadə olunur.

In [33]:
offline_row = offline_result.iloc[0]
online_row = online_result.iloc[0]

assert offline_row["transaction_count"] == online_row["transaction_count"]
assert offline_row["total_amount"] == online_row["total_amount"]
assert pd.isna(offline_row["avg_amount"]) and pd.isna(online_row["avg_amount"])

print("SUCCESS: Offline and Online features are consistent!")

SUCCESS: Offline and Online features are consistent!


##  Offline və Online Feature Consistency

Offline və Online Store-dan əldə edilmiş eyni müştəriyə aid feature-lər müqayisə edilir. `assert` yoxlamaları göstəricilərin uyğunluğunu təsdiqləyir.

##  Conclusion

Feast Feature Store PaySim datasetindən istifadə edilərək quruldu. Tarixi feature-lər Offline Store-da saxlanıldı, Online Store-dan müştəri üzrə feature retrieval həyata keçirildi və hər iki mənbədən əldə edilən nəticələrin uyğunluğu təsdiqləndi.